<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/ray_tune_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ray Tune Hyperparameter Sweep

Run hyperparameter tuning sweeps directly on a Colab GPU using [Ray Tune](https://docs.ray.io/en/latest/tune/index.html).

**Key features:**
- **ASHA scheduler** for early stopping of underperforming trials
- **Single-stage sweeps** — tune one curriculum stage at a time
- **Per-species reward signal search spaces** — automatically includes the correct reward parameters for each species and stage
- **Google Drive persistence** — all results, models, and checkpoints saved to Drive
- **Warm-start support** — load a checkpoint from a prior stage or prior sweep
- **Cross-session resume** — if your Colab session terminates mid-sweep, experiment state is restored from Drive on the next session so completed trials are kept and only partial trials restart

**Recommended runtime:** Colab Pro+ with A100 GPU (more CPU cores for MuJoCo vectorized envs)

**GPU-specific settings:**

| Setting | A100 (40GB) | L4 (24GB) | T4 (16GB) |
|---|---|---|---|
| `MAX_CONCURRENT` | 3 | 2 | 1 |
| `N_ENVS` | 8 | 4 | 4 |
| `NUM_TRIALS` | 20 | 16 | 12 |
| `TIMESTEPS_PER_TRIAL` | 4M | 3M | 2M |
| `ppo_batch_size` choices | 64–512 | 64–256 | 64–128 |
| `ppo_n_steps` choices | 1024–4096 | 1024–2048 | 1024–2048 |
| `ppo_net_arch` | all 6 | drop deep variants | small/medium/tapered |
| SAC `MAX_CONCURRENT` | 2 | 1 | 1 |
| Estimated sweep time | ~4–6h | ~6–8h | ~8–12h |

**Per-stage early stopping settings:**

| Setting | Stage 1 (balance) | Stage 2 (locomotion) | Stage 3 (behavior) |
|---|---|---|---|
| `GRACE_PERIOD` | 20 (~1M steps) | 40 (~2M steps) | 40 (~2M steps) |
| `REDUCTION_FACTOR` | 2 | 2 | 2 |
| `COLLAPSE_MIN_EVALS` | 8 | 12 | 12 |
| `COLLAPSE_PATIENCE` | 5 | 5 | 5 |

- **Stage 1:** Dense, immediate reward signal (alive bonus, posture). Trials that can't balance within ~1M steps are unlikely to recover.
- **Stage 2:** Curriculum ramp delays the locomotion reward signal (~500k ramp + warm-start adaptation). Needs more time before pruning.
- **Stage 3:** Similar to Stage 2 — species-specific behavior rewards (strike, bite) need time to emerge alongside reduced locomotion weights.

**Resuming an interrupted sweep (works across Colab sessions):**
1. Re-run Sections 1–2 (Setup, Configuration) with `RESUME = True`
2. The sweep directory is auto-detected (or set `RESUME_SWEEP_DIR` manually)
3. Section 2 automatically restores Ray Tune experiment state from Google Drive
4. Run Section 5–6 — completed trials are kept, partial trials restart from scratch

**How cross-session resume works:** During the sweep, an `ExperimentStateSyncCallback` periodically copies Ray Tune's experiment metadata from `/tmp/` to Google Drive. On resume, Section 2 copies it back to `/tmp/` so that `Tuner.restore()` can find all completed trial results.

## 1. Setup & Installation

In [ ]:
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    if importlib.util.find_spec("ray") is None:
        get_ipython().system('pip install -q "ray[tune]>=2.9.0"')

    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally — ensure ray[tune], mujoco, stable-baselines3 are installed.")

In [ ]:
import json
import logging
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import mujoco

from environments.shared.config import load_all_stages
from environments.shared.species_registry import get_species_config
from environments.shared.scripts.sweep import (
    NET_ARCH_PRESETS,
    build_search_space,
    detect_gpu_model,
    save_search_space,
    to_ray_tune,
)

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

In [ ]:
# ===== Species & Algorithm =====
SPECIES = "velociraptor"  # @param ["velociraptor", "brachiosaurus", "trex"]
ALGORITHM = "ppo"         # @param ["ppo", "sac"]

# ===== Sweep Stage =====
STAGE = 1                 # @param {type:"integer"} — curriculum stage (1, 2, or 3)

# ===== Ray Tune Budget =====
# See GPU settings table in the intro cell. For L4: NUM_TRIALS=16, MAX_CONCURRENT=2 (PPO) or 1 (SAC).
NUM_TRIALS = 50           # @param {type:"integer"} — total trials to run
MAX_CONCURRENT = 3        # @param {type:"integer"} — concurrent trials (3 for A100, 2 for L4, 1 for T4/SAC)
TIMESTEPS_PER_TRIAL = 6_000_000  # @param {type:"integer"} — training timesteps per trial (3M for L4)

# ===== Training =====
N_ENVS = 8                # @param {type:"integer"} — parallel envs per trial
SEED = 42                 # @param {type:"integer"}
EVAL_FREQ = 50_000        # @param {type:"integer"} — how often to evaluate (also used as ASHA report interval)

# ===== Scheduler =====
# True: use ASHA early stopping (prune underperforming trials).
# False: use FIFO scheduler (all trials run to completion — pure random search).
USE_ASHA = True           # @param {type:"boolean"}

# ===== ASHA Early Stopping (only used when USE_ASHA=True) =====
# See per-stage recommendations in the intro cell.
# Stage 1: grace_period=20, Stage 2+3: grace_period=40
GRACE_PERIOD = 30         # @param {type:"integer"} — minimum reports before ASHA can stop a trial
REDUCTION_FACTOR = 2      # @param {type:"integer"} — keep top 1/N trials at each rung

# ===== Reward Collapse Early Stopping =====
# Stops a trial if eval reward drops significantly from its peak (within a single trial).
# This is independent of ASHA (which compares trials against each other).
COLLAPSE_MIN_EVALS = 8    # @param {type:"integer"} — minimum evals before collapse detection activates
COLLAPSE_PATIENCE = 5     # @param {type:"integer"} — consecutive drops below threshold before stopping

# ===== Warm-start (optional) =====
# Path to a model checkpoint to load (e.g. from a prior stage).
# Leave empty to train from scratch.
LOAD_PATH = ""  # @param {type:"string"}

# ===== Resume (optional) =====
# Set to True to resume a previously interrupted sweep.
# Set RESUME_SWEEP_DIR to the Drive path of the previous sweep, or leave
# empty to auto-detect the latest sweep for this species/stage/algorithm.
RESUME = False            # @param {type:"boolean"}
RESUME_SWEEP_DIR = ""     # @param {type:"string"}

# ===== Google Drive =====
USE_GOOGLE_DRIVE = True   # @param {type:"boolean"}

print(f"Species:    {SPECIES}")
print(f"Algorithm:  {ALGORITHM.upper()}")
print(f"Stage:      {STAGE}")
print(f"Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"Scheduler:  {'ASHA' if USE_ASHA else 'FIFO (no pruning)'}")
if USE_ASHA:
    print(f"ASHA:       grace_period={GRACE_PERIOD}, reduction_factor={REDUCTION_FACTOR}")
print(f"Collapse:   min_evals={COLLAPSE_MIN_EVALS}, patience={COLLAPSE_PATIENCE}")
print(f"Load path:  {LOAD_PATH or '(none — training from scratch)'}")
print(f"Resume:     {RESUME}{f' from {RESUME_SWEEP_DIR}' if RESUME and RESUME_SWEEP_DIR else ''}")

In [ ]:
# ============================================================
# Google Drive Storage
# ============================================================
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/mesozoic-labs")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Results will persist to: {DRIVE_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    DRIVE_BASE = repo_root / "logs"
else:
    DRIVE_BASE = repo_root / "logs"
    print(f"Using local storage: {DRIVE_BASE}")

# ── Sweep output directory (on Drive for persistence) ──────────────────
if RESUME:
    # Resume: reuse an existing sweep directory
    if RESUME_SWEEP_DIR:
        SWEEP_DIR = Path(RESUME_SWEEP_DIR)
    else:
        # Auto-detect: find the latest sweep directory for this species/stage/algorithm
        _sweep_parent = DRIVE_BASE / "ray_tune_sweeps" / SPECIES
        _prefix = f"stage{STAGE}_{ALGORITHM}_"
        _candidates = sorted(
            [d for d in _sweep_parent.iterdir() if d.is_dir() and d.name.startswith(_prefix)],
            key=lambda d: d.name,
            reverse=True,
        ) if _sweep_parent.exists() else []
        assert _candidates, (
            f"RESUME=True but no previous sweep found in {_sweep_parent} "
            f"matching '{_prefix}*'. Set RESUME_SWEEP_DIR explicitly."
        )
        SWEEP_DIR = _candidates[0]
    assert SWEEP_DIR.exists(), f"Resume sweep directory does not exist: {SWEEP_DIR}"
    _sweep_name = SWEEP_DIR.name
    print(f"Resuming sweep from: {SWEEP_DIR}")
else:
    # Fresh sweep: create a new timestamped directory
    SWEEP_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    _sweep_name = f"stage{STAGE}_{ALGORITHM}_{SWEEP_TIMESTAMP}"
    SWEEP_DIR = DRIVE_BASE / "ray_tune_sweeps" / SPECIES / _sweep_name
    SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Sweep output directory: {SWEEP_DIR}")

# ── Local storage for Ray Tune internals ───────────────────────────────
# Ray Tune's checkpoint validation writes fail on Google Drive FUSE mounts.
# Use a fast local path for Ray's internal storage, and sync best models
# to Drive periodically for crash resilience.
LOCAL_RAY_DIR = Path("/tmp/ray_tune_storage") / _sweep_name
LOCAL_RAY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_TRIALS_DIR = Path("/tmp/ray_tune_trials") / _sweep_name
LOCAL_TRIALS_DIR.mkdir(parents=True, exist_ok=True)

# ── Restore Ray experiment state from Drive on resume ──────────────────
# When a Colab session terminates, /tmp/ is wiped. The experiment state is
# periodically synced to Drive during the sweep (via ExperimentStateSyncCallback).
# Restore it here so Tuner.restore() can find completed trial metadata.
if RESUME:
    import shutil as _shutil_restore

    _experiment_name = f"{SPECIES}_stage{STAGE}_{ALGORITHM}"
    _drive_experiment_dir = SWEEP_DIR / "ray_results" / _experiment_name
    _local_experiment_dir = LOCAL_RAY_DIR / _experiment_name

    if _drive_experiment_dir.exists() and not _local_experiment_dir.exists():
        print(f"Restoring Ray experiment state from Drive: {_drive_experiment_dir}")
        _shutil_restore.copytree(
            str(_drive_experiment_dir),
            str(_local_experiment_dir),
            dirs_exist_ok=True,
        )
        print(f"  Restored to: {_local_experiment_dir}")
    elif _local_experiment_dir.exists():
        print(f"Local experiment state already exists: {_local_experiment_dir}")
    else:
        print(
            f"Warning: No experiment state found on Drive at {_drive_experiment_dir}. "
            f"If this is the first resume attempt, the previous sweep may not have "
            f"synced its state. A fresh sweep will be started instead."
        )

print(f"Local Ray storage:    {LOCAL_RAY_DIR}")
print(f"Local trials dir:     {LOCAL_TRIALS_DIR}")
print(f"Drive sync target:    {SWEEP_DIR}")

## 3. Load Species & Stage Configs

In [ ]:
SPECIES_CFG = get_species_config(SPECIES)
STAGE_CONFIGS = load_all_stages(SPECIES)

env = SPECIES_CFG.env_class()
print(f"Environment: {SPECIES_CFG.env_class.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} — {cfg['description']}")
env.close()

## 4. Search Space

The search space is loaded from a JSON config file (`configs/sweep_ppo.json` or
`configs/sweep_sac.json`) so you can adjust parameter ranges without touching Python code.

Each config file uses a per-stage layout where job settings (`trials`, `timesteps`, etc.)
live alongside parameter specs. Only entries with a `"type"` key are treated as search-space
parameters — everything else is metadata.

To customise the search space, either:
1. **Edit the JSON config** in `configs/sweep_{algorithm}.json` directly, or
2. **Pass a custom config path** via `SWEEP_CONFIG_PATH` below.

The resolved search space is saved to Google Drive for record keeping.

**Per-stage rationale:**
- **Stage 1 (balance):** `alive_bonus` is the dominant reward signal — worth sweeping alongside species-specific stability rewards (posture, nosedive, drift, etc.)
- **Stage 2 (locomotion):** `alive_bonus` must stay low to avoid standing-trap; sweep forward velocity, curriculum ramp, and species-specific locomotion rewards
- **Stage 3 (behavior):** Sweep species-specific attack/goal rewards (strike, bite, food_reach) alongside reduced locomotion weights

In [ ]:
# Optional: override the default config file path.
# Leave empty to use configs/sweep_{algorithm}.json from the repo root.
SWEEP_CONFIG_PATH = ""  # @param {type:"string"}

_config_path = SWEEP_CONFIG_PATH or None  # None → use default

# Build the search space from the JSON config file.
_search_space_dict = build_search_space(SPECIES, STAGE, ALGORITHM, config_path=_config_path)
SEARCH_SPACE = to_ray_tune(_search_space_dict)

print(f"Search space for Stage {STAGE} / {SPECIES} ({ALGORITHM.upper()}): {len(SEARCH_SPACE)} params")
for name in SEARCH_SPACE:
    print(f"  {name}")

# Save the resolved search space to the sweep directory on Google Drive
# so each run has a permanent record of exactly which ranges were used.
_saved_path = save_search_space(
    _search_space_dict,
    SWEEP_DIR,
    species=SPECIES,
    stage=STAGE,
    algorithm=ALGORITHM,
    gpu_model=detect_gpu_model(),
    max_concurrent=MAX_CONCURRENT,
    n_envs=N_ENVS,
    timesteps_per_trial=TIMESTEPS_PER_TRIAL,
    num_trials=NUM_TRIALS,
    eval_freq=EVAL_FREQ,
    seed=SEED,
    use_asha=USE_ASHA,
    grace_period=GRACE_PERIOD,
    reduction_factor=REDUCTION_FACTOR,
    collapse_min_evals=COLLAPSE_MIN_EVALS,
    collapse_patience=COLLAPSE_PATIENCE,
)
print(f"\nSearch space saved for record keeping: {_saved_path}")

## 5. Trial Training Function

Each Ray Tune trial runs a function from `environments.shared.scripts.sweep.ray_tune`.
It applies the sampled hyperparameters to the TOML stage config, trains using the
project's existing infrastructure, and reports `best_mean_reward` plus a Ray checkpoint
to Ray Tune at each evaluation round.

In [ ]:
from environments.shared.scripts.sweep.ray_tune import train_trial

print("Trial function loaded from library.")

## 6. Run the Sweep

Ray Tune manages the trial scheduling. Two scheduler modes are available
(controlled by `USE_ASHA` in Section 2):

- **ASHA** (`USE_ASHA=True`): Stops underperforming trials early based on
  intermediate `best_mean_reward` reports, saving compute.
- **FIFO** (`USE_ASHA=False`): Pure random search — all trials run to completion
  with no pruning. Useful for collecting unbiased data across the full search space.

**ASHA parameters (when enabled):**
- `grace_period`: Minimum number of reports before a trial can be stopped (default 30 ≈ 1.5M steps)
- `reduction_factor`: Keep top ~50% of trials at each rung

**Note:** If you see `TuneError: insufficient resources`, reduce `MAX_CONCURRENT` in Section 2.

In [ ]:
import os

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler, FIFOScheduler

from environments.shared.scripts.sweep.ray_tune import (
    DriveProgressLogCallback,
    ExperimentStateSyncCallback,
    TrialTerminationCallback,
)

os.environ["TUNE_WARN_EXCESSIVE_EXPERIMENT_CHECKPOINT_SYNC_THRESHOLD_S"] = "0"

if ray.is_initialized():
    ray.shutdown()

ray.init(ignore_reinit_error=True)
print(f"Ray initialized: {ray.cluster_resources()}")

# Scheduler: ASHA for early stopping, or FIFO for pure random search (no pruning)
max_reports = TIMESTEPS_PER_TRIAL // EVAL_FREQ
if USE_ASHA:
    scheduler = ASHAScheduler(
        metric="best_mean_reward",
        mode="max",
        max_t=max_reports,
        grace_period=GRACE_PERIOD,
        reduction_factor=REDUCTION_FACTOR,
    )
    _scheduler_desc = f"ASHA (grace={GRACE_PERIOD}, reduction={REDUCTION_FACTOR})"
else:
    scheduler = FIFOScheduler()
    _scheduler_desc = "FIFO (no pruning — all trials run to completion)"

trial_callback = TrialTerminationCallback(report_interval_s=300)

# Sync callback: periodically copies Ray experiment state to Drive so that
# cross-session resume works even if the Colab session terminates mid-sweep.
_experiment_name = f"{SPECIES}_stage{STAGE}_{ALGORITHM}"
_local_experiment_dir = LOCAL_RAY_DIR / _experiment_name
_drive_ray_results_dir = SWEEP_DIR / "ray_results"

state_sync_callback = ExperimentStateSyncCallback(
    local_experiment_dir=_local_experiment_dir,
    drive_ray_results_dir=_drive_ray_results_dir,
    sync_interval_s=300,
)

# Progress log callback: writes a trial_progress.csv to Drive on each trial
# completion so you can check sweep progress even when the notebook is
# disconnected. Unlike collected_results.csv (written post-sweep with full
# evaluation metrics), this is updated incrementally with ASHA-reported metrics.
progress_log_callback = DriveProgressLogCallback(drive_sweep_dir=SWEEP_DIR)

# Ray Train v2 decoupled RunConfig from Tune — use the Tune-namespaced
# versions for Tuner configuration to avoid Train v2 deprecation errors.
from ray.tune import RunConfig as TuneRunConfig
from ray.tune import CheckpointConfig as TuneCheckpointConfig

# Fixed parameters passed to every trial (prefixed with _ to distinguish from search space)
fixed_config = {
    "_species": SPECIES,
    "_algorithm": ALGORITHM,
    "_stage": STAGE,
    "_timesteps": TIMESTEPS_PER_TRIAL,
    "_n_envs": N_ENVS,
    "_seed": SEED,
    "_eval_freq": EVAL_FREQ,
    "_load_path": LOAD_PATH or "",
    "_local_trials_dir": str(LOCAL_TRIALS_DIR),
    "_drive_sweep_dir": str(SWEEP_DIR),
    "_collapse_min_evals": COLLAPSE_MIN_EVALS,
    "_collapse_patience": COLLAPSE_PATIENCE,
}

full_config = {**fixed_config, **SEARCH_SPACE}

print(f"\nStarting Ray Tune sweep:")
print(f"  Species:    {SPECIES}")
print(f"  Stage:      {STAGE}")
print(f"  Algorithm:  {ALGORITHM.upper()}")
print(f"  Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"  Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"  Params:     {len(SEARCH_SPACE)} search-space dimensions")
print(f"  Scheduler:  {_scheduler_desc}")
print(f"  Collapse:   min_evals={COLLAPSE_MIN_EVALS}, patience={COLLAPSE_PATIENCE}")
print(f"  Local:      {LOCAL_RAY_DIR}")
print(f"  Drive:      {SWEEP_DIR}")
print(f"  Progress:   {SWEEP_DIR / 'trial_progress.csv'}")
if LOAD_PATH:
    print(f"  Warm-start: {LOAD_PATH}")

In [ ]:
import shutil as _shutil

from environments.shared.scripts.sweep.ray_tune import collect_ray_results
from environments.shared.scripts.sweep.results import plot_sweep_results, write_results_csv
from environments.shared.scripts.sweep.scoring import compute_quality_scores

# Allocate fractional GPU per trial to prevent OOM with concurrent trials
_gpu_fraction = 1.0 / max(MAX_CONCURRENT, 1)
_trainable = tune.with_resources(train_trial, {"cpu": 2, "gpu": _gpu_fraction})

if RESUME and _local_experiment_dir.exists():
    print(f"Restoring Tuner from: {_local_experiment_dir}")
    print("  Completed trials will be kept; partial trials will restart from scratch.")
    tuner = tune.Tuner.restore(
        path=str(_local_experiment_dir),
        trainable=_trainable,
        resume_unfinished=True,
        resume_errored=True,
        param_space=full_config,
    )
elif RESUME:
    print(
        "Warning: RESUME=True but no experiment state found locally or on Drive. "
        "Starting a fresh sweep instead."
    )
    RESUME = False  # fall through to fresh sweep

if not RESUME:
    tuner = tune.Tuner(
        _trainable,
        param_space=full_config,
        tune_config=tune.TuneConfig(
            scheduler=scheduler,
            num_samples=NUM_TRIALS,
            max_concurrent_trials=MAX_CONCURRENT,
        ),
        run_config=TuneRunConfig(
            name=f"{SPECIES}_stage{STAGE}_{ALGORITHM}",
            storage_path=str(LOCAL_RAY_DIR),
            checkpoint_config=TuneCheckpointConfig(num_to_keep=2),
            verbose=0,
            callbacks=[trial_callback, state_sync_callback, progress_log_callback],
        ),
    )

results = tuner.fit()
print("\nSweep complete!")

# ── Final sync of Ray Tune experiment state to Drive ─────────────────
# This ensures the final state (all trials completed) is on Drive,
# complementing the periodic syncs that happened during the sweep.
if _local_experiment_dir.exists():
    print(f"Syncing final Ray results to Drive: {_drive_ray_results_dir}")
    try:
        _shutil.copytree(
            str(_local_experiment_dir),
            str(_drive_ray_results_dir / _local_experiment_dir.name),
            dirs_exist_ok=True,
        )
        print("  Ray results synced to Drive successfully.")
    except OSError as e:
        print(f"  Warning: Drive sync of Ray results failed: {e}")

## 7. Results & Analysis

In [ ]:
import pandas as pd

# Get results as a DataFrame
results_df = results.get_dataframe()

# Sort by best reward
if "best_mean_reward" in results_df.columns:
    results_df = results_df.sort_values("best_mean_reward", ascending=False)

# Display the hyperparameter columns (filter out internal Ray columns)
param_cols = [c for c in results_df.columns if c.startswith(("ppo_", "sac_", "env_", "curriculum_"))]
metric_cols = ["best_mean_reward", "last_mean_reward", "timesteps"]
display_cols = [c for c in metric_cols + param_cols if c in results_df.columns]

print(f"\n{'=' * 60}")
print(f"Sweep Results: {SPECIES} Stage {STAGE} ({ALGORITHM.upper()})")
print(f"{'=' * 60}")
display(results_df[display_cols].head(20))

In [ ]:
# Best trial details
best_result = results.get_best_result(metric="best_mean_reward", mode="max")

print(f"\nBest Trial:")
print(f"  best_mean_reward: {best_result.metrics.get('best_mean_reward', 'N/A'):.4f}")
print(f"  Hyperparameters:")
for key, value in best_result.config.items():
    if not key.startswith("_"):
        print(f"    {key}: {value}")

In [ ]:
from environments.shared.scripts.sweep.ray_tune import collect_ray_results
from environments.shared.scripts.sweep.results import plot_sweep_results, write_results_csv
from environments.shared.scripts.sweep.scoring import compute_quality_scores

# ── Standardized sweep CSV ──
_sweep_rows = collect_ray_results(results_df, STAGE, STAGE_CONFIGS[STAGE])

# Apply quality scoring (post-analysis ranking — does not affect training objective)
compute_quality_scores(_sweep_rows, STAGE)

collected_csv = write_results_csv(_sweep_rows, SWEEP_DIR / "collected_results.csv")
print(f"Standardized CSV saved to: {collected_csv}")

# ── Sweep analysis plots ──
plot_sweep_results(collected_csv, SPECIES, ALGORITHM, save_dir=SWEEP_DIR)
print(f"Sweep analysis plots saved to: {SWEEP_DIR}")

# Also save the raw Ray Tune DataFrame for Ray-specific columns
raw_csv = SWEEP_DIR / "sweep_results_raw.csv"
results_df.to_csv(str(raw_csv), index=False)
print(f"Raw Ray Tune CSV saved to: {raw_csv}")

## 7b. Post-Sweep Deep Analysis (Standalone)

Run this section to evaluate the **top trials** with full locomotion metrics
(forward velocity, gait symmetry, cost of transport, etc.) that are **not**
captured during the Ray Tune sweep itself.

**This section is self-contained** — if your Colab session restarted after the
sweep, just re-run Sections 1–3 (Setup, Configuration, Load Species) and then
set `SWEEP_DIR` below to the Drive path from the previous sweep. You do **not**
need to re-run the Tune sweep.

In [ ]:
# ============================================================
# Standalone reload: set this to a previous sweep's Drive path
# if your Colab session restarted after the sweep finished.
# When running right after Section 6, leave as-is — SWEEP_DIR
# is already set.
# ============================================================
# SWEEP_DIR = Path("/content/drive/MyDrive/mesozoic-labs/ray_tune_sweeps/velociraptor/stage2_ppo_20250101_120000")  # uncomment & edit

TOP_K = 5  # @param {type:"integer"} — number of top trials to analyze
EVAL_EPISODES = 30  # @param {type:"integer"} — episodes per trial evaluation

print(f"Sweep directory: {SWEEP_DIR}")
print(f"Analyzing top {TOP_K} trials with {EVAL_EPISODES}-episode evaluation")

# ── Discover trial directories ─────────────────────────────────────────
trials_root = SWEEP_DIR / "trials"
assert trials_root.exists(), f"No trials directory found at {trials_root}"

# Find trial dirs that have a best_model.zip and rank by evaluations.npz.
# If evaluations.npz is missing (e.g. session crashed before it was synced
# to Drive), fall back to a quick 1-episode eval to rank the trial.
_trial_dirs = sorted(trials_root.iterdir())
_valid_trials = []
_missing_npz_count = 0
for td in _trial_dirs:
    if td.is_dir() and (td / "models" / "best_model.zip").exists():
        # Read best_mean_reward from evaluations.npz
        eval_npz = td / "evaluations.npz"
        reward = float("-inf")
        if eval_npz.exists():
            _eval_data = np.load(str(eval_npz))
            _mean_per_eval = _eval_data["results"].mean(axis=1)
            reward = float(_mean_per_eval.max())
        else:
            _missing_npz_count += 1
        _valid_trials.append((td, reward))

# If most trials are missing evaluations.npz, run a quick 1-episode eval
# to get approximate rankings instead of sorting everything to -inf.
_have_npz = sum(1 for _, r in _valid_trials if r != float("-inf"))
if _valid_trials and _have_npz < len(_valid_trials) // 2:
    print(f"\nNote: {_missing_npz_count}/{len(_valid_trials)} trials missing evaluations.npz (not synced to Drive).")
    print("Running quick 1-episode ranking eval for trials without evaluations.npz...")
    from stable_baselines3 import PPO as _PPO_rank, SAC as _SAC_rank
    from environments.shared.train_base import _ensure_sb3 as _ensure_sb3_rank

    _sb3_rank = _ensure_sb3_rank()
    _stage_cfg_rank = STAGE_CONFIGS[STAGE]
    _env_kwargs_rank = _stage_cfg_rank["env_kwargs"].copy()
    _alg_cls_rank = _SAC_rank if ALGORITHM == "sac" else _PPO_rank

    _reranked = []
    for td, reward in _valid_trials:
        if reward != float("-inf"):
            _reranked.append((td, reward))
            continue
        model_path = str(td / "models" / "best_model")
        vecnorm_path = td / "models" / "best_model_vecnorm.pkl"
        try:
            def _mk():
                return _sb3_rank["Monitor"](SPECIES_CFG.env_class(**_env_kwargs_rank))
            _ev = _sb3_rank["DummyVecEnv"]([_mk])
            if vecnorm_path.exists():
                _ev = _sb3_rank["VecNormalize"].load(str(vecnorm_path), _ev)
                _ev.training = False
                _ev.norm_reward = False
            _m = _alg_cls_rank.load(model_path, env=_ev)
            obs = _ev.reset()
            total = 0.0
            while True:
                action, _ = _m.predict(obs, deterministic=True)
                obs, rews, dones, _ = _ev.step(action)
                total += float(rews[0])
                if dones[0]:
                    break
            _ev.close()
            _reranked.append((td, total))
        except Exception as e:
            print(f"  Warning: Could not eval {td.name}: {e}")
            _reranked.append((td, float("-inf")))
    _valid_trials = _reranked
    print("Quick ranking complete.")
elif _missing_npz_count > 0:
    print(f"\nNote: {_missing_npz_count} trial(s) missing evaluations.npz — ranked last.")

_valid_trials.sort(key=lambda x: x[1], reverse=True)
_top_trials = _valid_trials[:TOP_K]

print(f"\nFound {len(_valid_trials)} trials with saved models, analyzing top {len(_top_trials)}:")
for i, (td, r) in enumerate(_top_trials):
    _r_str = f"{r:.2f}" if r != float("-inf") else "N/A (quick eval failed)"
    print(f"  {i+1}. {td.name}  (best_mean_reward = {_r_str})")

In [ ]:
import warnings

import pandas as pd
from stable_baselines3 import PPO, SAC

from environments.shared.evaluation import eval_policy
from environments.shared.metrics import LocomotionMetrics
from environments.shared.reporting import (
    build_stage_results_from_eval_data,
    generate_stage_artifacts,
)
from environments.shared.train_base import create_vec_env, _ensure_sb3

_logger = logging.getLogger(__name__)

sb3 = _ensure_sb3()
stage_config = STAGE_CONFIGS[STAGE]
env_kwargs = stage_config["env_kwargs"].copy()
alg_cls = SAC if ALGORITHM == "sac" else PPO

# ── Evaluate each top trial ───────────────────────────────────────────
analysis_rows = []

for rank, (trial_dir, sweep_reward) in enumerate(_top_trials, 1):
    trial_name = trial_dir.name
    model_dir = trial_dir / "models"
    model_path = str(model_dir / "best_model")
    vecnorm_path = str(model_dir / "best_model_vecnorm.pkl")

    print(f"\n{'=' * 60}")
    print(f"Trial {rank}/{len(_top_trials)}: {trial_name}  (sweep reward: {sweep_reward:.2f})")
    print(f"{'=' * 60}")

    # Load model + VecNormalize
    def _make_eval_env():
        return sb3["Monitor"](SPECIES_CFG.env_class(**env_kwargs))

    eval_env = sb3["DummyVecEnv"]([_make_eval_env])
    if Path(vecnorm_path).exists():
        eval_env = sb3["VecNormalize"].load(vecnorm_path, eval_env)
        eval_env.training = False
        eval_env.norm_reward = False

    model = alg_cls.load(model_path, env=eval_env)

    # Full evaluation with LocomotionMetrics
    episode_reports = []
    for ep in range(EVAL_EPISODES):
        obs = eval_env.reset()
        metrics = LocomotionMetrics()
        total_reward = 0.0
        step = 0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = eval_env.step(action)
            total_reward += float(rewards[0])
            step += 1
            metrics.record_step(infos[0], float(rewards[0]))
            if dones[0]:
                break
        episode_reports.append(metrics.compute())

    eval_env.close()

    agg = LocomotionMetrics.aggregate_episodes(episode_reports)

    row = {
        "rank": rank,
        "trial": trial_name,
        "sweep_reward": round(sweep_reward, 2),
        "eval_reward": round(agg.get("mean_total_reward", 0), 2),
        "eval_reward_std": round(agg.get("std_total_reward", 0), 2),
        "ep_length": round(agg.get("mean_episode_length", 0), 1),
        "fwd_vel_m/s": round(agg.get("mean_mean_forward_velocity", 0), 3),
        "fwd_vel_std": round(agg.get("std_mean_forward_velocity", 0), 3),
        "max_fwd_vel": round(agg.get("mean_max_forward_velocity", 0), 3),
        "distance_m": round(agg.get("mean_total_distance", 0), 3),
        "vel_consistency": round(agg.get("mean_velocity_consistency", 0), 3),
        "gait_symmetry": round(agg.get("mean_gait_symmetry", 0), 3),
        "stride_freq_Hz": round(agg.get("mean_stride_frequency", 0), 3),
        "cost_of_transport": round(agg.get("mean_cost_of_transport", 0), 4),
        "tilt_rad": round(agg.get("mean_mean_tilt_angle", 0), 3),
        "pelvis_height_m": round(agg.get("mean_mean_pelvis_height", 0), 3),
    }
    analysis_rows.append(row)

    _logger.info(
        "  reward=%.2f  fwd_vel=%.3f m/s  distance=%.2f m  symmetry=%.3f  CoT=%.4f",
        row["eval_reward"], row["fwd_vel_m/s"], row["distance_m"],
        row["gait_symmetry"], row["cost_of_transport"],
    )

    # ── Generate stage artifacts (training curves, summary, videos) ────
    # Check for evaluations.npz and diagnostics.npz — these are needed for
    # training curve and diagnostics graphs. If missing, warn and skip graphs
    # but still generate the summary and videos.
    _has_eval_npz = (trial_dir / "evaluations.npz").exists()
    _has_diag_npz = (trial_dir / "diagnostics.npz").exists()

    if not _has_eval_npz:
        warnings.warn(
            f"Trial {trial_name}: evaluations.npz missing (not synced to Drive). "
            f"Skipping training curve graphs for this trial.",
            stacklevel=1,
        )
    if not _has_diag_npz and _has_eval_npz:
        warnings.warn(
            f"Trial {trial_name}: diagnostics.npz missing. "
            f"Diagnostics graphs will be incomplete.",
            stacklevel=1,
        )

    # Build a stage_results dict from the LocomotionMetrics eval so that
    # the stage summary matches the training notebook's output.
    # Start from on-disk eval data (best eval checkpoint metrics from
    # evaluations.npz) if available, then enrich with the live eval above.
    if _has_eval_npz:
        _stage_results = build_stage_results_from_eval_data(
            trial_dir, STAGE, stage_config, timesteps=TIMESTEPS_PER_TRIAL,
        )
    else:
        # Minimal stage_results when evaluations.npz is missing
        _stage_results = {
            "stage": STAGE,
            "name": stage_config["name"],
            "description": stage_config["description"],
            "timesteps": TIMESTEPS_PER_TRIAL,
            "duration_seconds": 0,
        }

    # Enrich with the live LocomotionMetrics eval (same as training notebook)
    _sim_dt = stage_config["env_kwargs"].get("sim_dt", 0.01)
    _stage_results.update({
        "mean_reward": row["eval_reward"],
        "std_reward": row["eval_reward_std"],
        "mean_episode_length": row["ep_length"],
        "std_episode_length": round(agg.get("std_episode_length", 0), 1),
        "mean_forward_vel": row["fwd_vel_m/s"],
        "std_forward_vel": row["fwd_vel_std"],
        "mean_distance_traveled": row["distance_m"],
        "sim_dt": _sim_dt,
        "model_path": model_path,
        "vecnorm_path": vecnorm_path,
        # Best model eval section (uses the same eval since we loaded the best model)
        "best_model_reward": row["eval_reward"],
        "best_model_std_reward": row["eval_reward_std"],
        "best_model_length": row["ep_length"],
        "best_model_std_length": round(agg.get("std_episode_length", 0), 1),
        "best_model_fwd_vel": row["fwd_vel_m/s"],
        "best_model_std_fwd_vel": row["fwd_vel_std"],
        "best_model_success_rate": round(agg.get("mean_success_rate", 0), 4),
        "best_model_n_episodes": EVAL_EPISODES,
    })

    generate_stage_artifacts(
        SPECIES_CFG,
        stage_config,
        STAGE,
        ALGORITHM,
        stage_dir=trial_dir,
        seed=SEED,
        stage_results=_stage_results,
        timesteps=TIMESTEPS_PER_TRIAL,
        record_videos=True,
        generate_graphs=_has_eval_npz,
    )

print(f"\nPost-sweep evaluation complete for {len(analysis_rows)} trials.")

In [ ]:
from environments.shared.scripts.sweep.scoring import compute_quality_scores
from environments.shared.visualization import plot_trial_comparison

# ── Quality scoring ────────────────────────────────────────────────────
# Apply stage-aware weighted scoring to rank trials beyond raw reward.
# Scoring config is loaded from configs/quality_scoring.toml.
# This does NOT affect the training objective (which remains best_mean_reward).
compute_quality_scores(analysis_rows, STAGE)

# ── Comparison table (sorted by quality_score) ─────────────────────────
analysis_df = pd.DataFrame(analysis_rows)

# Re-index by quality_rank if scoring was applied
if "quality_rank" in analysis_df.columns and analysis_df["quality_rank"].notna().all():
    analysis_df = analysis_df.set_index("quality_rank")
    analysis_df.index.name = "quality_rank"
else:
    analysis_df = analysis_df.set_index("rank")

print(f"\n{'=' * 80}")
print(f"Post-Sweep Analysis: {SPECIES.title()} Stage {STAGE} ({ALGORITHM.upper()}) — Top {len(analysis_rows)} Trials")
print(f"(Ranked by quality_score — weighted composite of locomotion metrics)")
print(f"{'=' * 80}\n")
display(analysis_df)

# Save to CSV
analysis_csv = SWEEP_DIR / "post_sweep_analysis.csv"
analysis_df.to_csv(str(analysis_csv))
print(f"\nAnalysis CSV saved to: {analysis_csv}")

# ── Comparison plots (shared with Vertex AI sweep) ────────────────────
comparison_path = SWEEP_DIR / "post_sweep_comparison.png"
plot_trial_comparison(
    analysis_rows,
    species=SPECIES,
    stage=STAGE,
    save_path=comparison_path,
    show=True,
)
print(f"Comparison plot saved to: {comparison_path}")

## 8. Save Results to Google Drive

Persist the sweep results CSV, best trial config, and analysis plots to Google Drive
so they survive session restarts.

In [ ]:
# Save best trial config as JSON
best_config = {k: v for k, v in best_result.config.items() if not k.startswith("_")}
best_config_with_meta = {
    "species": SPECIES,
    "algorithm": ALGORITHM,
    "stage": STAGE,
    "best_mean_reward": best_result.metrics.get("best_mean_reward"),
    "timesteps_per_trial": TIMESTEPS_PER_TRIAL,
    "num_trials": NUM_TRIALS,
    "scheduler": "ASHA" if USE_ASHA else "FIFO",
    "hyperparameters": best_config,
}

best_config_path = SWEEP_DIR / "best_trial_config.json"
with open(str(best_config_path), "w") as f:
    json.dump(best_config_with_meta, f, indent=2, default=str)
print(f"Best trial config saved to: {best_config_path}")

# Copy best trial's model to a convenient location on Drive.
# Models may be in local trial dir or already synced to Drive.
best_trial_id = best_result.metrics.get("trial_id", "")
_local_model_dir = LOCAL_TRIALS_DIR / str(best_trial_id) / "models"
_drive_model_dir = SWEEP_DIR / "trials" / str(best_trial_id) / "models"

# Prefer local (always complete), fall back to Drive (synced copy)
best_trial_model_dir = _local_model_dir if _local_model_dir.exists() else _drive_model_dir

best_model_dest = SWEEP_DIR / "best_model"
best_model_dest.mkdir(parents=True, exist_ok=True)

import shutil

_found_any = False
for src_pattern in [f"stage{STAGE}_final.zip", f"stage{STAGE}_final_vecnorm.pkl",
                    "best_model.zip", "best_model_vecnorm.pkl"]:
    src_files = list(best_trial_model_dir.glob(src_pattern)) if best_trial_model_dir.exists() else []
    for src in src_files:
        dest = best_model_dest / src.name
        shutil.copy2(str(src), str(dest))
        print(f"  Copied: {src.name} -> {dest}")
        _found_any = True

if not _found_any:
    print(f"  Warning: No model files found in {best_trial_model_dir}")
    print(f"  Check if the best trial completed successfully.")

print(f"\nAll results saved to: {SWEEP_DIR}")
print(f"\nTo use the best model for the next stage, set LOAD_PATH to:")
print(f"  {best_model_dest / 'best_model'}")

## 9. Apply Best Hyperparameters

Use this cell to generate `--override` flags for the CLI training scripts,
so you can retrain with the best hyperparameters found by the sweep.

In [ ]:
# Generate CLI override flags for the best trial
overrides = []
for key, value in best_config.items():
    for prefix in ("ppo", "sac", "env", "curriculum"):
        if key.startswith(prefix + "_"):
            param = key[len(prefix) + 1:]
            if param == "net_arch":
                # net_arch is handled specially by the training script
                overrides.append(f"{prefix}.policy_kwargs.net_arch={value}")
            elif isinstance(value, float) and value == int(value):
                overrides.append(f"{prefix}.{param}={int(value)}")
            else:
                overrides.append(f"{prefix}.{param}={value}")
            break

override_str = " ".join(f"--override {o}" for o in overrides)

print(f"Best hyperparameters as CLI overrides:\n")
print(f"python -m environments.{SPECIES}.scripts.train_sb3 train \\")
print(f"    --stage {STAGE} \\")
print(f"    --algorithm {ALGORITHM} \\")
print(f"    --timesteps {TIMESTEPS_PER_TRIAL} \\")
for o in overrides:
    print(f"    --override {o} \\")

## 10. Cleanup

In [ ]:
ray.shutdown()
print("Ray shutdown complete.")
print(f"\nSweep results directory: {SWEEP_DIR}")
print(f"Best trial config: {best_config_path}")
print(f"Best model: {best_model_dest}")